# Лабораторная №1 курса "Синтез Речи"

## EDA

### Импорты и константы

In [ ]:
import json
import re
import string
from collections import Counter, defaultdict
from pprint import pprint

import emoji
import pandas as pd
import regex
from babel.numbers import get_currency_symbol, list_currencies
from tqdm import tqdm


In [37]:
RUSLAN_METADATA_PATH = "../../data/metadata_RUSLAN_22200.csv"
RANDOM_STATE = 42

### Читаем RUSLAN метадату

In [38]:
RUSLAN_METADATA = pd.read_csv(RUSLAN_METADATA_PATH, sep="|", header=None, index_col=0)
RUSLAN_METADATA.head()

,1
0,
000000_RUSLAN,С тревожным чувством берусь я за перо.
000001_RUSLAN,Кого интересуют признания литературного неудач...
000002_RUSLAN,Что поучительного в его исповеди?
000003_RUSLAN,Да и жизнь моя лишена внешнего трагизма.
000004_RUSLAN,Я абсолютно здоров.


### Для сокращений и аббревиатур будем использовать [ГПНТБ словарь](https://www.gpntb.ru/2017-08-21-14-53-51/339-produkty-gpntb/7630-elektronnyj-slovar-standartizovannykh-sokrashchenij.html)

In [39]:
SHORT_FORMS_ABBREVIATIONS_PATH = "data/short_forms.dict"

In [40]:
short_forms_abbreviations_df = pd.read_csv(SHORT_FORMS_ABBREVIATIONS_PATH)
# Убираем все, кроме пар {сокращение: слово/словосочетание}
short_forms_abbreviations_df = short_forms_abbreviations_df[
    ["Слово (словосочетание)", "Сокращение"]
]
short_forms_abbreviations_df = short_forms_abbreviations_df.dropna()

#### Нормализуем пробелы

In [41]:
def normalize_spaces(input_string):
    return re.sub(r'\s+', ' ', input_string).strip()

In [42]:
short_forms_abbreviations_df["Слово (словосочетание)"] = short_forms_abbreviations_df[
    "Слово (словосочетание)"
].apply(normalize_spaces)

short_forms_abbreviations_df["Сокращение"] = short_forms_abbreviations_df[
    "Сокращение"
].apply(normalize_spaces)

short_forms_abbreviations_df.head(5)

,Слово (словосочетание),Сокращение
0,например,напр.
1,сравнение; сравни,ср.
2,Германская Демократическая Республика,ГДР
3,Всемирный совет мира,ВСМ
4,Всемирная организация здравоохранения,ВОЗ


#### Разделяем сокращения и аббревиатуры

In [43]:
def separate_short_forms_abbreviations(df: pd.DataFrame):
    """
    Разделять по КАПСУ нет смысла, есть одиночные сокращения типа `С`.
    Ну и плюс, вдруг там будут какие-нибудь сокращения капсом лучше брать так: 
    все, что с точкой на конце – сокращения; все, что нет – аббревиатуры
    """
    short_forms = {}
    abbreviations = {}
    for row in df.iterrows():
        candidate = row[1]["Сокращение"]
        if "." in candidate or "-" in candidate:
            short_forms[row[1]["Сокращение"]] = row[1]["Слово (словосочетание)"]
        else:
            abbreviations[row[1]["Сокращение"]] = row[1]["Слово (словосочетание)"]
    return short_forms, abbreviations

In [44]:
short_forms_dict, abbreviations_dict = separate_short_forms_abbreviations(
    short_forms_abbreviations_df
)

In [45]:
short_forms_list = short_forms_dict.keys()
abbreviations_list = abbreviations_dict.keys()

In [46]:
abbreviations_list

dict_keys(['ГДР', 'ВСМ', 'ВОЗ', 'США', 'ВФП', 'ВНР', 'ИСО', 'ИФЛА', 'ВМО', 'МОТ', 'ФАО', 'МАгАтЭ', 'МФД', 'НРБ', 'ООН', 'ЮНЕСКО', 'ЧССР', 'ПНР', 'ФРГ', 'ОЭСР', 'СРР', 'вуз', 'С', 'км', 'г', 'З', 'кВт', 'к/с', 'АО', 'ПВ', 'ч', 'сут', 'мфише҆', '№', 'мин', 'нп', 'К°; Ко', 'Ю', 'В', 'bokbd', 'bd', 'billedfonogr', 'AS', 'AD', 'AB', 'AG', 'Ko', 'ko', 'letop', 'ezds', 'FID', 'FIAB', 'dr', 'ČSSR', 'cm', 'FSM', 'IAEA', 'afr', 'ca', 'hbd', 'hung', 'hullbd', 'indledn', 'ISO', 'ILO', 'IFLA', 'FAO', 'mficha', 'mfiche', 'mfișă', 'Mfiche', 'mforma', 'mfiš', 'mfisza', 'NV', 'MNK', 'mtárs', 'Mij', 'OMM', 'OECD', 'OIT', 'OY', 'ps', 'PRL', 'OCDE', 'UNESCO', 'OMS', 'OAA', 'ко', 'мфиш', 'obst', 'nr', 'no', 'obál', 'Obst', 'st', 'SA', 'RSR', 'ONU', 'sloven', 'wg', 'Ztg', 'tr', 'Zsstellung', 'Zsfassung', 'broš', 'co', 'BRD', 'cuad', 'WFTU', 'WHO', 'WPC', 'WMO', 'db', 'diamficha', 'cie', 'DDR', 'CMP', 'см', 'цм', 'обра', 'АД', 'сие', 'вж', 'ВИ', 'РГ', 'UN', 'USA', 'ua', 'h', 'Hz', 'Pa', 'pc', 'H', 'д; d', 'p

#### Междометия

In [47]:
with open("data/interjections.json", encoding="utf-8") as f:
    interjections = json.load(f)

#### Множество валютных знаков и обозначений

In [48]:
currencies_code = list_currencies()
currencies_symbols = {
    get_currency_symbol(currency, locale="en_US")
    for currency in currencies_code
}

currencies = currencies_code | currencies_symbols

### Считаем статистику по фразам в корпусе RUSLAN

In [49]:
# Группы название – паттерн для типичных поисков
patterns = {
    "dates": [r"\b(?:0?[1-9]|[12]\d|3[01])[\./-](?:0?[1-9]|1[0-2])[\./-](?:\d{2}|\d{4})\b", "words"],
    "phone_numbers": [r"(?:\+7|8)[\s().-]*(?:\d[\s().-]*){10}", "words"],
    "ordinal_numbers": [r"\b\d+-?(?:й|ый|ой|я|ая|ое|ее|е|ые|ие|ых|их|ым|им|ом|ем|му|ему|ую|ю|го|ого|ей|ими|ыми|ми|м|и)\b", "chars"],
    "technical": [r"[*|/@+<>=\\^~_{}\[\]]", "chars"],
    "punct_runs": [r"(?<![.!?])(?!\?!|\?.\.|\.\.\.)[.!?]{2,}", "chars"],
    "space_before_comma": [r"\s+,", "chars"],
    "space_after_comma": [r"(,(?!\s)|,(?=\s{2,}))", "chars"],
    "spaces_run": [r"\s{2,}", "chars"],
    "wrong_quot_marks": [r"[“”„‟‹›„““”]", "chars"],
    "unfinished_quot_marks": [r"(\"(=!.+\")|\'(=!.+\')|«(=!.+»))", "chars"],
    "russian_smiling_parenthesis": [r"^[^(]*\)+", "chars"],
}

#### Класс-счетчик

In [ ]:
class EntryStatistics:
    def __init__(self, input_string):
        self.input_string = input_string
        self.input_string_preprocessed = self.preprocess(self.input_string)
        self.input_string_tokenized = self.input_string.split()
        self.input_string_preprocessed_tokenized = self.input_string_preprocessed.split()

        self.len_chars = len(self.input_string)
        self.len_words = len(self.input_string_tokenized)

        self.dict_of_counters = self.build_dict_of_counters()
        self.vector = self.build_vector()

    def build_dict_of_counters(self):
        return {
            # Типичные
            "dates": self.regex_find(patterns["dates"][0]),
            "phone_numbers": self.regex_find(patterns["phone_numbers"][0]),
            "ordinal_numbers": self.regex_find(patterns["ordinal_numbers"][0]),
            "technical": self.regex_find(patterns["technical"][0]),
            "punct_runs": self.regex_find(patterns["punct_runs"][0]),
            "space_before_comma": self.regex_find(patterns["space_before_comma"][0]),
            "space_after_comma": self.regex_find(patterns["space_after_comma"][0]),
            "spaces_run": self.regex_find(patterns["spaces_run"][0]),
            "wrong_quot_marks": self.regex_find(patterns["wrong_quot_marks"][0]),
            "unfinished_quot_marks": self.regex_find(patterns["unfinished_quot_marks"][0]),
            "russian_smiling_parenthesis": self.regex_find(patterns["russian_smiling_parenthesis"][0]),
            # Нетипичные
            "numerals": self.get_numerals(),
            "currency": self.get_currency(),
            "non_cyrillic": self.get_non_cyrillic(),
            "short_forms": self.get_short_forms(),
            "special_symbols": self.get_special_symbols(),
            "abbreviations": self.get_abbreviations(),
            "interjections": self.get_interjections(),
            "emoji": self.get_emoji()
        }
    
    def build_vector(self):
        return [
            sum(counter.values()) / (self.len_chars + 1) 
            for counter in self.dict_of_counters.values()
        ]
    
    def preprocess(self, input_string):
        input_string = input_string.lower()
        input_string = input_string.translate(str.maketrans("", "", string.punctuation))
        return input_string

    def regex_find(self, pattern) -> Counter:
        matches = re.findall(pattern, self.input_string)
        matches_count = Counter(matches)

        return matches_count

# Нетипичные поиски
    def get_numerals(self) -> Counter:
        pattern = r'-?\d*\.?\d+'
        matches = re.findall(pattern, self.input_string)
        matches = [float(x) if '.' in x else int(x) for x in matches]
        matches_count = Counter(matches)

        return matches_count

    def get_currency(self):
        pattern = r"(?<!\w)(?:" + "|".join(
            re.escape(currency)
            for currency in sorted(currencies, key=len, reverse=True)
        ) + r")(?!\w)"
        matches = re.findall(pattern, self.input_string)
        matches_count = Counter(matches)

        return matches_count

    def get_non_cyrillic(self):
        matches = [
            char
            for char in self.input_string
            if regex.fullmatch(r"[\p{L}]", char)
            and not regex.fullmatch(r"\p{Cyrillic}", char)
        ]
        matches_count = Counter(matches)

        return matches_count
    
    def get_short_forms(self):
        input_string = self.input_string.lower()

        short_forms = [
            (short_form, len(short_form))
            for short_form in short_forms_list
        ]

        matches = []

        i = 0
        while i < len(input_string):
            found = None
            found_length = 0
            for short_form, short_form_length in short_forms:
                if (
                    input_string.startswith(short_form, i)
                    and short_form_length > found_length
                ):
                    found = short_form
                    found_length = short_form_length
            if found is not None:
                matches.append(found)
                i += found_length
            else:
                i += 1
        matches_count = Counter(matches)

        return matches_count

    def get_abbreviations(self):
        def preprocess_punct(input_string):
            return input_string.translate(str.maketrans("", "", string.punctuation))
        
        tokens = [
            preprocess_punct(token)
            for token in self.input_string_tokenized
        ]

        matches = []
        for abbreviation in abbreviations_list:
            if abbreviation in tokens:
                matches.append(abbreviation)
        matches_count = Counter(matches)

        return matches_count

    def get_special_symbols(self):
        pattern = r'.,!?"\';…:\-—«»\u0301*|/@+<>=\\^~_{}\[\]'
        # \u0301 = " ́"
        matches = regex.findall(rf"[^\p{{L}}\p{{N}}\s{pattern}]", self.input_string)
        matches_count = Counter(matches)

        return matches_count

    def get_interjections(self):
        STRIP = string.punctuation + "…«»—"
        tokens = {token.strip(STRIP).lower() for token in self.input_string_tokenized}
        matches = [i for i in interjections if i.lower() in tokens]
        matches_count = Counter(matches)

        return matches_count

    def get_emoji(self):
        matches = emoji.emoji_list(self.input_string)
        matches = [match["emoji"] for match in matches]
        matches_count = Counter(matches)

        return matches_count

#### Статистика

* топ-10 самых распространенных;
* встречаемость на символ/слово;
* встречаемость на фразу;
* 

In [51]:
stats = defaultdict(Counter)

In [52]:
for phrase in RUSLAN_METADATA.iterrows():
    entry_stats = EntryStatistics(phrase[1][1]).dict_of_counters
    
    for param, counter in entry_stats.items():
        stats[param] += counter

##### Цифры

In [124]:
pprint(stats, width=60, sort_dicts=False)

defaultdict(<class 'collections.Counter'>,
            {'dates': Counter(),
             'phone_numbers': Counter(),
             'ordinal_numbers': Counter(),
             'technical': Counter({'/': 4,
                                   '*': 1,
                                   '<': 1,
                                   '>': 1}),
             'punct_runs': Counter({'!..': 372,
                                    '..': 16,
                                    '?.': 2,
                                    '!!!': 2,
                                    '??': 1,
                                    '!.': 1,
                                    '.?': 1}),
             'space_before_comma': Counter(),
             'space_after_comma': Counter({',': 5}),
             'spaces_run': Counter({'  ': 62,
                                    '   ': 2,
                                    '     ': 1}),
             'wrong_quot_marks': Counter({'„': 72,
                                          '“': 57,
 

## Классификатор нормализован-не нормализован

### Иморты и константы

In [54]:
import warnings
from pathlib import Path

import joblib
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import StandardScaler
from torch import Tensor
from transformers import AutoModel, AutoTokenizer

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/home/ivan/workspace/tts_labs_itmo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Как и зачем аугментируем?

В изначальном `dev_sentences.csv` 300 примеров из приблизительно одного домена: клиентская поддержка, телеком и подобное. Поэтому экспериментально аугментируем: просим Claude (Opus 5) и ChatGPT (5.6 Luna) нагенерировать по 1000 примеров каждый в `dev_sentences_train.csv` по такому же правилу, что и оригинальные, но с другой тематикой. 

Смотрим на то, на каких классификаторах такая аугментация дает прирост, а где нет.

Важно, чтобы:

* Признаки, определяющие, нормализован/не нормализован текст, были схвачены корректно;
* Не создавались "лишние" признаки и взаимосвязи. Допустим, если мы хотим отледить `)` в конце предложения, то нужно обязательно добавить в аугментированные обучающие данные примеры с парами скобок;
* Модель не скатывалась в конструирование повторяющихся фраз с изменением плейсхолдеров:
    `Мама мыла раму с утра`,
    `Папа мыл раму в 12`,
    `В г. Г г-н мыл раму`
и так далее.

In [55]:
DEV_TRAIN_PATH = "data/dev_sentences_train.csv"
DEV_TEST_PATH = "data/dev_sentences.csv"

In [56]:
DEV_TRAIN = pd.read_csv(DEV_TRAIN_PATH, sep="|")
DEV_TEST = pd.read_csv(DEV_TEST_PATH, sep="|")

### Эмбеддинги

In [57]:
sents_train = DEV_TRAIN["text"].to_list()
sents_test = DEV_TEST["text"].to_list()

In [58]:
labels_train = DEV_TRAIN["is_normalized"].to_list()
labels_test = DEV_TEST["is_normalized"].to_list()

In [59]:
def average_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    last_hidden = last_hidden_states.masked_fill(~attention_mask[..., None].bool(), 0.0)
    return last_hidden.sum(dim=1) / attention_mask.sum(dim=1)[..., None]

In [60]:
sents_train = [f"query: {sent}" for sent in sents_train]
sents_test = [f"query: {sent}" for sent in sents_test]

In [61]:
tokenizer = AutoTokenizer.from_pretrained(
    'intfloat/multilingual-e5-large',
    )
model = AutoModel.from_pretrained(
    'intfloat/multilingual-e5-large',
    torch_dtype=torch.float16,
    ).to(device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 391/391 [00:01<00:00, 220.79it/s]


In [62]:
def embed(sentences):
    # Токенизация
    batch_dict = tokenizer(
        sentences, max_length=512, padding=True, truncation=True, return_tensors='pt'
    )

    batch_dict = {
        k: v.to(device)
        for k, v in batch_dict.items()
    }

    with torch.inference_mode():
        outputs = model(**batch_dict)

        embeddings = average_pool(
            outputs.last_hidden_state, batch_dict['attention_mask']
        )

        # Нормализация эмбеддингов
        embeddings = F.normalize(embeddings, p=2, dim=1)

    return embeddings.to("cpu")

In [63]:
embeddings_train = []
embeddings_test = []

In [64]:
batch_size = 8

In [65]:
for i in tqdm(range(0, len(sents_train), batch_size)):
    embeddings = embed(sents_train[i : i + batch_size])
    embeddings_train.extend(embeddings)

100%|██████████| 252/252 [01:36<00:00,  2.62it/s]


In [66]:
for i in tqdm(range(0, len(sents_test), batch_size)):
    embeddings = embed(sents_test[i : i + batch_size])
    embeddings_test.extend(embeddings)

100%|██████████| 38/38 [00:14<00:00,  2.64it/s]


In [67]:
print(embeddings_train[0].shape)

torch.Size([1024])


In [68]:
torch.save(embeddings_train, "data/embeddings_weights/dev_sentences_emb_train.pt")
torch.save(embeddings_test, "data/embeddings_weights/dev_sentences_emb_test.pt")

In [69]:
if device == "cuda":
    torch.cuda.empty_cache()

##### С аугментацией трейна

In [70]:
embeddings_train = torch.load("data/embeddings_weights/dev_sentences_emb_train.pt")
embeddings_test = torch.load("data/embeddings_weights/dev_sentences_emb_test.pt")

In [71]:
labels_train = DEV_TRAIN["is_normalized"].to_list()
labels_test = DEV_TEST["is_normalized"].to_list()

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5
)

logreg_model.fit(embeddings_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [73]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(embeddings_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       0.89      0.86      0.88       156
           1       0.85      0.89      0.87       144

    accuracy                           0.87       300
   macro avg       0.87      0.87      0.87       300
weighted avg       0.87      0.87      0.87       300

Лучшие параметры: {'C': 100, 'class_weight': 'balanced', 'max_iter': 250, 'solver': 'liblinear', 'warm_start': True}


##### Без аугментации трейна

Берем изначальные 300 примеров с лейблами

In [75]:
labels = DEV_TEST["is_normalized"].to_list()

In [76]:
embeddings = torch.load("data/embeddings_weights/dev_sentences_emb_test.pt")

In [77]:
embeddings_train, embeddings_test, \
    labels_train, labels_test = train_test_split(embeddings, labels, test_size=0.3, random_state=RANDOM_STATE)

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5
)

logreg_model.fit(embeddings_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [79]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(embeddings_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       0.87      0.82      0.85        50
           1       0.79      0.85      0.82        40

    accuracy                           0.83        90
   macro avg       0.83      0.83      0.83        90
weighted avg       0.84      0.83      0.83        90

Лучшие параметры: {'C': 1000, 'class_weight': None, 'max_iter': 250, 'solver': 'liblinear', 'warm_start': True}


### CountVectorizer

#### С аугментацией трейна

In [81]:
sents_train = DEV_TRAIN["text"].to_list()
sents_test = DEV_TEST["text"].to_list()

In [82]:
labels_train = DEV_TRAIN["is_normalized"].to_list()
labels_test = DEV_TEST["is_normalized"].to_list()

In [83]:
count_vectorizer = CountVectorizer(analyzer="char_wb", lowercase=True, ngram_range=(1, 3))

count_vectorizer.fit_transform(sents_train)
count_vectorizer.transform(sents_test)

count_vectors_train = count_vectorizer.transform(sents_train)
count_vectors_test = count_vectorizer.transform(sents_test)

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5
)

logreg_model.fit(count_vectors_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [85]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(count_vectors_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       0.93      0.85      0.89       156
           1       0.85      0.93      0.89       144

    accuracy                           0.89       300
   macro avg       0.89      0.89      0.89       300
weighted avg       0.89      0.89      0.89       300

Лучшие параметры: {'C': 1, 'class_weight': 'balanced', 'max_iter': 250, 'solver': 'liblinear', 'warm_start': True}


#### Без аугментации трейна

In [87]:
sents = DEV_TEST["text"].to_list()
labels = DEV_TEST["is_normalized"].to_list()

In [88]:
sents_train, sents_test, \
    labels_train, labels_test = train_test_split(sents, labels, test_size=0.3, random_state=RANDOM_STATE)

In [89]:
count_vectorizer = CountVectorizer(analyzer="char_wb", lowercase=True, ngram_range=(1, 3))

count_vectorizer.fit_transform(sents_train)
count_vectorizer.transform(sents_test)

count_vectors_train = count_vectorizer.transform(sents_train)
count_vectors_test = count_vectorizer.transform(sents_test)

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5
)

logreg_model.fit(count_vectors_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [91]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(count_vectors_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       0.81      0.78      0.80        50
           1       0.74      0.78      0.76        40

    accuracy                           0.78        90
   macro avg       0.78      0.78      0.78        90
weighted avg       0.78      0.78      0.78        90

Лучшие параметры: {'C': 10, 'class_weight': None, 'max_iter': 250, 'solver': 'lbfgs', 'warm_start': True}


### Собственные вектора

#### С аугментацией трейна

In [93]:
sents_train = DEV_TRAIN["text"].to_list()
sents_test = DEV_TEST["text"].to_list()

In [94]:
labels_train = DEV_TRAIN["is_normalized"].to_list()
labels_test = DEV_TEST["is_normalized"].to_list()

In [95]:
feature_vectors_train = [EntryStatistics(entry).vector for entry in sents_train]
feature_vectors_test = [EntryStatistics(entry).vector for entry in sents_test]

In [96]:
scaler = StandardScaler()
scaler.fit(feature_vectors_train)

feature_vectors_train = scaler.transform(feature_vectors_train)
feature_vectors_test = scaler.transform(feature_vectors_test)

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5,
)

logreg_model.fit(feature_vectors_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [98]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(feature_vectors_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       1.00      0.92      0.96       156
           1       0.92      1.00      0.96       144

    accuracy                           0.96       300
   macro avg       0.96      0.96      0.96       300
weighted avg       0.96      0.96      0.96       300

Лучшие параметры: {'C': 10, 'class_weight': None, 'max_iter': 250, 'solver': 'lbfgs', 'warm_start': True}


In [99]:
Path("data/classifiers/feature_vectors").mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_best_model, "data/classifiers/feature_vectors/logreg_augmented.joblib")

['data/classifiers/feature_vectors/logreg_augmented.joblib']

#### Без аугментации трейна

In [100]:
sents = DEV_TEST["text"].to_list()
labels = DEV_TEST["is_normalized"].to_list()

In [101]:
sents_train, sents_test, \
    labels_train, labels_test = train_test_split(sents, labels, test_size=0.3, random_state=RANDOM_STATE)

In [102]:
feature_vectors_train = [EntryStatistics(entry).vector for entry in sents_train]
feature_vectors_test = [EntryStatistics(entry).vector for entry in sents_test]

In [103]:
scaler = StandardScaler()
scaler.fit(feature_vectors_train)

feature_vectors_train = scaler.transform(feature_vectors_train)
feature_vectors_test = scaler.transform(feature_vectors_test)

In [ ]:
logreg_params = {
    "C": [0.01, 0.1, 1, 10, 100, 1000],
    "solver": ["lbfgs", "liblinear"],
    "max_iter": [250, 500, 1000],
    "class_weight": [None, "balanced"],
    "warm_start": [True, False],
}

logreg_model = GridSearchCV(
    LogisticRegression(random_state=RANDOM_STATE),
    logreg_params,
    scoring="f1_macro",
    cv=5
)

logreg_model.fit(feature_vectors_train, labels_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'class_weight': [None, 'balanced'], 'max_iter': [250, 500, ...], 'solver': ['lbfgs', 'liblinear'], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1_macro'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",None
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verb

In [105]:
logreg_best_model = logreg_model.best_estimator_
logreg_preds = logreg_best_model.predict(feature_vectors_test)

print(classification_report(labels_test, logreg_preds))
print(f"Лучшие параметры: {logreg_model.best_params_}")

              precision    recall  f1-score   support

           0       1.00      0.86      0.92        50
           1       0.85      1.00      0.92        40

    accuracy                           0.92        90
   macro avg       0.93      0.93      0.92        90
weighted avg       0.93      0.92      0.92        90

Лучшие параметры: {'C': 100, 'class_weight': None, 'max_iter': 250, 'solver': 'lbfgs', 'warm_start': True}


In [106]:
Path("data/classifiers/feature_vectors").mkdir(parents=True, exist_ok=True)

joblib.dump(logreg_best_model, "data/classifiers/feature_vectors/logreg_non_augmented.joblib")

['data/classifiers/feature_vectors/logreg_non_augmented.joblib']